# Real-Time Endpoint

Deploys the model to an always-on HTTPS endpoint for low-latency, on-demand inference.

<b>STEP 1:</b> Get a training job <br> <br>
<b>STEP 2:</b> Attach it to an estimator <br> <br>
<b>STEP 3:</b> Deploy the estimator (returns a predictor) <br> <br>
<b>STEP 4:</b> Use the predictor for inference<br> <br> 

2 Methods for inference - Using a predictor and using sagemaker runtime directly

In [ ]:
import boto3
import sagemaker

REGION = "us-east-1"

boto_session = boto3.Session(region_name=REGION)
sm_session = sagemaker.Session(boto_session=boto_session)

## 1. Attach to the trained estimator

Use an existing training job and attach to an estimator. (Ref: https://sagemaker.readthedocs.io/en/v2.16.0/api/training/estimators.html)

In [ ]:
from sagemaker.estimator import Estimator

TRAINING_JOB_NAME = "sagemaker-xgboost-2026-08-31-09-05-39-925"

estimator = Estimator.attach(TRAINING_JOB_NAME, sagemaker_session=sm_session)
print(f"Using training job {TRAINING_JOB_NAME}")

In [ ]:
# Debug helpers, left commented -- uncomment to stream training logs or inspect
# the estimator's attributes if `.attach()` above doesn't behave as expected.
#estimator.logs()

## 2. Deploy

Built-in XGBoost expects/returns CSV, so we pair the deploy call with a CSV
serializer/deserializer — that's what lets `predictor.predict()` accept a plain Python
list/DataFrame row instead of hand-formatted request bytes.

In [ ]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

# Deploy the model to a SageMaker endpoint. Change this to a unique name if you want to deploy multiple endpoints.
endpoint_name = "fraud-detection-realtime-new"

predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=endpoint_name,
    serializer=CSVSerializer(),
    deserializer=CSVDeserializer(),
)
print(f"Endpoint deployed successfully: {predictor.endpoint_name}")

## 3. Invoking the endpoint

### Method 1 - Via the SDK's `predictor.predict()`

`Predictor.predict()` is a thin wrapper. It serializes the input using `CSVSerializer` and deserializes the output using `CSVDeserializer`. It internally calls sagemaker runtime client's `invoke_endpoint`

In [ ]:
# describe_training_job gives you the full job metadata (hyperparameters, data channels,
# instance type, metrics, artifact path) -- handy when you only have a job name and need
# to reconstruct what was actually run. Uncomment the next cell to print it.
sm_client = boto_session.client("sagemaker")
desc = sm_client.describe_training_job(TrainingJobName=TRAINING_JOB_NAME)

In [ ]:
# A single row of features, in the same column order used in training (no target column).
sample_input = ['6854.53', '0', '0', '0', '0', '1', '0']

### 3a. Correct Input

In [ ]:
result = predictor.predict(sample_input)
print(result[0][0])

### 3a. error case 1 — malformed data (non-numeric value)

Fails at CSV-to-float parsing, before XGBoost's missing-value handling ever gets involved.

In [ ]:
bad_input = ['6854.53', '0', '0', "hello", '0', '1', '0']  # "hello" where a number is expected

try:
    result = predictor.predict(bad_input)
    print(result)
except Exception as e:
    print(f"Error: {e}")

### 3a. error case 2 — nonexistent endpoint

A client/infrastructure error, not a model error — this fails before any container is
reached.

In [ ]:
from sagemaker.predictor import Predictor

incorrect_endpoint_predictor = Predictor(
    endpoint_name="endpoint-123",
    sagemaker_session=sm_session,
    serializer=CSVSerializer(),
    deserializer=CSVDeserializer(),
)

try:
    result = incorrect_endpoint_predictor.predict(sample_input)
    print(result)
except Exception as e:
    print(f"Error: {e}")

### 3b. Via raw `boto3` (`sagemaker-runtime`) — what the SDK call above does under the hood,
useful when invoking from a service that isn't a Python/SageMaker-SDK environment


<b>IMPORTANT :</b> `predictor.predict()` and a direct `runtime.invoke_endpoint()` call are not two different code paths with different rules. `Predictor.predict()` is a thin wrapper. t serializes the input, then calls
`self.sagemaker_session.sagemaker_runtime_client.invoke_endpoint(...)`, the exact same boto3 call made directly below.

In [ ]:
runtime = boto_session.client("sagemaker-runtime")

### 3b. correct case

In [ ]:
csv_body = ",".join(str(v) for v in sample_input)

print(csv_body)

In [ ]:
sample_input

In [ ]:
# Input is comma seperated values

csv_body = ",".join(str(v) for v in sample_input)

response = runtime.invoke_endpoint(
    EndpointName=predictor.endpoint_name, # Name of deployed endpoint
    ContentType="text/csv", #type of input
    Body=csv_body, # actual input
)


In [ ]:
print(response["Body"].read().decode())

### 3b. error case 1 — malformed data (non-numeric value)

Same underlying `invoke_endpoint` call as 3a's error case 1 — same result.

In [ ]:
bad_input = ['6854.53', '0', '0', "hello", '0', '1', '0']  # "hello" where a number is expected
bad_body = ",".join(str(v) for v in bad_input)

try:
    response = runtime.invoke_endpoint(
        EndpointName=predictor.endpoint_name,
        ContentType="text/csv",
        Body=bad_body,
    )
    print(response["Body"].read().decode())
except Exception as e:
    print(f"Error: {e}")

### 3b. error case 2 — nonexistent endpoint

Same as 3a's error case 2, but via a direct `invoke_endpoint` call rather than a `Predictor`.

In [ ]:
try:
    runtime.invoke_endpoint(
        EndpointName="this-endpoint-does-not-exist",
        ContentType="text/csv",
        Body="1,2,3",
    )
except Exception as e:
    print(f"Error: {e}")

## 4. Cleanup — delete the endpoint

**IMPORTANT : Run this before moving on.** `predictor.delete_endpoint()` removes both the endpoint and
its endpoint configuration, so nothing is left billing in the background.

In [ ]:
predictor.delete_endpoint()
print("Endpoint deleted.")